# Weight-Space Reading: a Controlled Lab

**Goal.** Separate a model's learned visual features from the names attached to its output indices. This uses simple synthetic RGB data and does **not** reproduce the published CNN/LLM results. Do not interpret any of the following as a Claude or Astra evaluation.


In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()/"src"))
import numpy as np
from weight_audit.core import CLASS_NAMES, train_classifier, dataset, map_labels_to_new_indices, tiny_forward_exercise
from weight_audit.experiment import run


## 1. Train a simple classifier

This classifier has no hidden layers, so visualizing its weights is a positive control for directly interpretable features.


In [ ]:
model = train_classifier(seed=7)
x, y = dataset(100, seed=107)
print("Test accuracy:", model.accuracy(x,y))


## 2. Permute final-layer columns, but update the semantic label map

The original numeric output coordinates are reordered. Correctly mapping those coordinates back to the original class meanings preserves predictions.


In [ ]:
new_to_old=np.array([2,0,3,1])
remapped=model.permute_outputs(new_to_old)
y_remapped=map_labels_to_new_indices(y,new_to_old)
print("Original accuracy", model.accuracy(x,y))
print("Permuted accuracy with updated class map", remapped.accuracy(x,y_remapped))
print("Correct new label order", remapped.class_names)
assert np.allclose(remapped.logits(x), model.logits(x)[:,new_to_old])


## 3. Show the class-order prior's failure

A hypothetical observer who attaches the old class names to the new output indices gets zero of four correct. This demonstrates the experimental intervention, **not** a measured failure of any LLM.


In [ ]:
naive_index_accuracy=np.mean([CLASS_NAMES[i]==remapped.class_names[i] for i in range(4)])
print("Fixed-index prior label accuracy:", naive_index_accuracy)


## 4. Negative control: shuffle the weight vectors

Scrambling input-feature positions inside each output class retains histograms but destroys spatial structure. Unlike a label permutation this actually changes the learned function.


In [ ]:
shuffled=model.shuffle_weights(seed=38)
print("Shuffled weight test accuracy", shuffled.accuracy(x,y))


## 5. A true weight-flow calculation

Ask an LLM to compute the actual hidden preactivations, ReLU outputs, and final logits from the tiny 2→3→2 network in `prompts/05_exact_forward_pass.txt`. Tools should be disabled. The answer key below must be withheld until after its prediction.


In [ ]:
tiny_forward_exercise()


## 6. Export independent model-audit prompts

Run each prompt in a new chat with the same model and identical decoding settings. Score numerical observations and semantic class names separately, repeat across seeds, and retain transcripts.


In [ ]:
metrics=run(Path("results"),seed=7)
metrics


## 7. Hidden neuron permutation: identical function, different weights
A network can have different internal weight coordinates and compute exactly the same function. This complicates naive neuron-index interpretation.


In [ ]:
from weight_audit.core import hidden_unit_symmetry
hidden_unit_symmetry()


## Interpretation

This lab demonstrates a causal control for class-name index priors. It does not determine how an LLM internally performs the interpretation. For real mechanistic interpretability, test exact forward passes, unseen architectures and datasets, and counterfactual inputs.
